# 08 · OpenSearch walkthrough

This is an ungraded deep dive into **Amazon OpenSearch Service**, this lab's *vector store A*. In its Serverless form it's also the default vector store behind Bedrock Knowledge Bases. Agents rely on it for **keyword and semantic retrieval at scale**, with filters and per-index security.

| Section | What you'll see |
|---|---|
| 1. Serverless anatomy | Collections, the three security policies behind every 403, SigV4, OCUs |
| 2. The index | `knn_vector` mapping (engine, space type, HNSW), text analyzers, Serverless quirks |
| 3. Ingestion | Bulk loading, eventual consistency, why document ids are ignored, upsert by business key |
| 4. BM25 | Analyzers in action, unbounded scores |
| 5. kNN | Vector scores, and why they can't be added to BM25 |
| 6. Filtered kNN | Post-filtering versus efficient (in-graph) filtering, for tenants and metadata |
| 7. Hybrid | Client-side RRF (goal 4) versus a server-side search pipeline |
| 8. Agentic patterns | A retrieval tool built for grounding, with the tenant taken from trusted context |

Everything happens in a scratch index `wt-clauses` in the lab's collection, which the last cell deletes. The lab's index `coa_chunks_knn` is never touched. It uses the same 40 clauses (and cached Titan embeddings) as notebook 07.

In [ ]:
import os, sys, json, time, math
sys.path.insert(0, os.path.abspath('.'))
import labkit, boto3
from opensearchpy import helpers
from src.opensearch_client import OpenSearchVectorStore

store = OpenSearchVectorStore()          # SigV4-signed client for the lab's collection, from .env
store.wait_until_ready()
os_client = store.client
IDX = 'wt-clauses'
ENGINE = os.environ.get('OPENSEARCH_ENGINE', 'faiss')
print('endpoint:', os.environ['OPENSEARCH_HOST'], '| service:', os.environ.get('OPENSEARCH_SERVICE'), '| engine:', ENGINE)

def show(resp, fields=('clause_id', 'tenant')):
    hits = resp['hits']['hits']
    for h in hits:
        s = h['_source']
        print(f"  {h['_score']:8.4f}  " + '  '.join(str(s.get(f)) for f in fields) + f"  {s['body'][:60]}")
    print(f'  ({len(hits)} hits)')

## 1 · Serverless anatomy

- **Collections.** A *collection* is the unit you create, and you choose its type: `SEARCH`, `TIMESERIES` or `VECTORSEARCH` (this lab). There's no cluster to size: capacity is measured in **OCUs** (OpenSearch Compute Units), which scale automatically, separately for indexing and search. That minimum OCU floor is the lab's main cost.
- **Three security policies.** Access needs all three:
  1. an **encryption** policy (an AWS-owned or customer KMS key);
  2. a **network** policy (public endpoint, or VPC endpoints only);
  3. a **data access** policy: which IAM principals may do what on which collections and indexes.
- **IAM and SigV4.** Every request is SigV4-signed with service name `aoss`. IAM needs `aoss:APIAccessAll`, and the data access policy must also list the principal. A **403** almost always means the principal is missing from the data access policy.

In [ ]:
aoss = boto3.client('opensearchserverless', region_name=labkit.REGION)
collection_id = os.environ['OPENSEARCH_HOST'].split('.')[0]
col = aoss.batch_get_collection(ids=[collection_id])['collectionDetails'][0]
print(f"collection {col['name']}  type={col['type']}  status={col['status']}  standbyReplicas={col.get('standbyReplicas')}")

for ptype in ('encryption', 'network'):
    for p in aoss.list_security_policies(type=ptype)['securityPolicySummaries']:
        doc = aoss.get_security_policy(name=p['name'], type=ptype)['securityPolicyDetail']['policy']
        if col['name'] in json.dumps(doc):
            print(f'\n{ptype} policy {p["name"]}:', json.dumps(doc))
for p in aoss.list_access_policies(type='data')['accessPolicySummaries']:
    doc = aoss.get_access_policy(name=p['name'], type='data')['accessPolicyDetail']['policy']
    if col['name'] in json.dumps(doc):
        print(f'\ndata access policy {p["name"]}:')
        for rule in doc:
            print('  principals:', rule['Principal'])
            for r in rule['Rules']:
                print('   ', r['ResourceType'], r['Resource'], r['Permission'])

## 2 · The index: mapping is a design decision

- **`knn_vector`.** Set the `dimension` (must equal the embedding size), the `space_type` (`cosinesimil`, `l2` or `innerproduct`) and the **engine**:
  - `faiss`: the default for Serverless vector collections, and the one used here;
  - `nmslib`: legacy;
  - `lucene`: managed domains only; Serverless doesn't support it, as `src/opensearch_client.py` notes.
- **HNSW parameters.** `m` and `ef_construction` mean the same as in pgvector (notebook 07).
- **Text fields.** A text field's **analyzer** decides what BM25 can match. The mapping below indexes `body` twice:
  - with the `english` analyzer (stemming, stop words);
  - as a sub-field `body.std` with the `standard` analyzer, like the lab's index uses.
- **Keyword fields** (`tenant`, `lob`, `clause_id`) are exact-match: they're what you filter on.

In [ ]:
clauses = labkit.corpus()
embs = labkit.embed([c['text'] for c in clauses])
DIMS = len(embs[0])

if os_client.indices.exists(index=IDX):
    os_client.indices.delete(index=IDX)
os_client.indices.create(index=IDX, body={
    'settings': {'index': {'knn': True}},
    'mappings': {'properties': {
        'clause_id': {'type': 'keyword'},
        'tenant':    {'type': 'keyword'},
        'lob':       {'type': 'keyword'},
        'body':      {'type': 'text', 'analyzer': 'english', 'fields': {'std': {'type': 'text', 'analyzer': 'standard'}}},
        'embedding': {'type': 'knn_vector', 'dimension': DIMS,
                      'method': {'name': 'hnsw', 'space_type': 'cosinesimil', 'engine': ENGINE,
                                 'parameters': {'m': 16, 'ef_construction': 128}}},
    }},
})
print(json.dumps(os_client.indices.get_mapping(index=IDX)[IDX]['mappings']['properties']['embedding'], indent=2))

## 3 · Ingestion on Serverless

Two Serverless behaviours that the lab's code works around:
- **No `refresh`.** New documents become searchable after a delay of up to about a minute, so ingestion code must *wait for visibility*. That's `wait_until_searchable` in the lab.
- **No client-chosen document ids** in vector collections. Every index call creates a new document. Re-ingesting the same chunk therefore creates a **duplicate**, which is why goal 4's RRF deduplicates by `chunk_id`.

Bulk-load the 40 clauses, then wait until they're all visible:

In [ ]:
actions = [{'_index': IDX, '_source': {'clause_id': c['clause_id'], 'tenant': c['tenant'], 'lob': c['lob'], 'body': c['text'], 'embedding': e}}
           for c, e in zip(clauses, embs)]
ok, errors = helpers.bulk(os_client, actions, raise_on_error=False)
print('bulk indexed:', ok, '| errors:', errors[:1])

def wait_for_count(expected, timeout=240):
    t0 = time.time()
    while time.time() - t0 < timeout:
        n = os_client.count(index=IDX)['count']
        if n == expected:
            print(f'{n} documents searchable after {time.time() - t0:.0f}s')
            return n
        time.sleep(5)
    print(f'timeout: {n} documents visible, expected {expected}')
    return n

wait_for_count(len(clauses))

### Idempotent re-ingestion: upsert by business key

You can't overwrite by id, so make the **business key** (`clause_id`) the identity:
1. find the documents that carry the key;
2. index the new version;
3. delete the old versions by their (server-assigned) `_id`.

The cell re-ingests CY-01 with updated text and takes a couple of minutes, because each step waits for visibility. It's optional.

In [ ]:
RUN_UPSERT_DEMO = True

def upsert_clause(doc):
    old = os_client.search(index=IDX, body={'size': 10, '_source': False, 'query': {'term': {'clause_id': doc['clause_id']}}})['hits']['hits']
    os_client.index(index=IDX, body=doc)                 # no id=: Serverless assigns one
    for h in old:
        os_client.delete(index=IDX, id=h['_id'])
    return [h['_id'] for h in old]

if RUN_UPSERT_DEMO:
    c = next(x for x in clauses if x['clause_id'] == 'CY-01')
    new_text = c['text'] + ' Payments to sanctioned groups are never reimbursed.'
    replaced = upsert_clause({'clause_id': c['clause_id'], 'tenant': c['tenant'], 'lob': c['lob'], 'body': new_text,
                              'embedding': labkit.embed([new_text])[0]})
    print('replaced _ids:', replaced)
    wait_for_count(len(clauses))                          # still 40: one in, one out
    time.sleep(30)
    hits = os_client.search(index=IDX, body={'query': {'term': {'clause_id': 'CY-01'}}})['hits']['hits']
    print('CY-01 copies:', len(hits), '| text:', hits[0]['_source']['body'][-55:] if hits else '-')

## 4 · BM25: what the analyzer lets you match

BM25 scores a document by matching **terms**, so what counts as "the same term" is decided by the analyzer:
- **`standard`** lower-cases text and splits it on word boundaries;
- **`english`** also removes stop words and **stems**: *flooding → flood*, *damages → damag*, *covered → cover*.

Run the same question against `body.std` and against `body`:

In [ ]:
Q = 'flooding damages covered'
for field in ('body.std', 'body'):
    print(f'\nmatch on {field}:')
    show(os_client.search(index=IDX, body={'size': 5, 'query': {'match': {field: Q}}}))

- **The standard analyzer** needs the exact surface form. Only documents containing *covered* (and nothing for *flooding*) match.
- **The english analyzer** matches *flood*, *damage* and *covered* in all their forms.
- **Scores are unbounded.** They depend on term rarity and document length, so the same document can score 3 for one query and 12 for another. Treat them as an ordering, not a probability.

The lab's index uses the default `standard` analyzer on `text`. That's why, in goal 4, the query word *exclusions* didn't match *exclusion* in the Nexus Tower chunk. Choose the analyzer to fit your corpus language and its jargon.

## 5 · kNN: vector scores are a different currency

In [ ]:
QV = 'Is my building protected if a river overflows?'     # no shared words with 'flood'
qv = labkit.embed([QV])[0]
resp = os_client.search(index=IDX, body={'size': 5, 'query': {'knn': {'embedding': {'vector': qv, 'k': 5}}}})
show(resp)

by_id = {c['clause_id']: e for c, e in zip(clauses, embs)}
print('\n  score  vs  cosine similarity (computed locally)')
for h in resp['hits']['hits']:
    e = by_id.get(h['_source']['clause_id'])
    if e:
        cos = sum(a * b for a, b in zip(qv, e))          # Titan vectors are normalised
        print(f"  {h['_score']:.4f}  vs  {cos:.4f}   {h['_source']['clause_id']}")

- The top hit should be CP-01 (*flood damage … excluded*) or AU-05, even though the question never says "flood". That's the semantic half of retrieval.
- OpenSearch turns a distance into a **positive score** with a formula that depends on the space type and engine. Compare the two columns: the order is the same, but the numbers aren't the cosine.
- So **never add kNN scores to BM25 scores**. One is roughly 0–1 or 0–2, the other unbounded. Fuse by **rank** (RRF), or normalise both first (section 7).

## 6 · Filtered kNN: tenants and metadata

An agent searching *one broker's* clauses must filter. Where the filter runs matters:
- **Post-filter.** Put `knn` in a `bool` query and the filter beside it. The engine finds the global top-k, **then** filters, so you can get fewer than k results.
- **Efficient filtering.** Put the `filter` **inside** the `knn` clause. The engine searches only among matching documents and returns k of them.

In [ ]:
QF = 'What is excluded from cover?'
qf = labkit.embed([QF])[0]
tenant_filter = {'term': {'tenant': 'harbor'}}

print('post-filter (bool: knn k=5 + filter):')
show(os_client.search(index=IDX, body={'size': 5, 'query': {'bool': {
    'must': [{'knn': {'embedding': {'vector': qf, 'k': 5}}}], 'filter': [tenant_filter]}}}))

print('\nefficient filter (filter inside knn):')
try:
    show(os_client.search(index=IDX, body={'size': 5, 'query': {'knn': {'embedding': {'vector': qf, 'k': 5, 'filter': tenant_filter}}}}))
except Exception as e:
    print('  not supported here:', str(e)[:200], '\n  fallback: over-fetch (k=50) with the bool post-filter, then keep the top 5')

This is the same lesson as pgvector's filtered search (notebook 07, section 4), where the fix was an *iterative* scan. Security-relevant filters such as the tenant must come from **trusted context**, never from the model's arguments. Section 8 builds a tool that way.

## 7 · Hybrid: client-side RRF versus a search pipeline

**Client-side RRF** (goal 4's approach) runs BM25 and kNN as two requests and fuses them by rank in your code. It works on every engine and every version, and the logic is visible.

In [ ]:
NO_VECTORS = {'excludes': ['embedding']}        # don't ship 1024 floats back with every hit

def bm25(q, k, tenant=None):
    query = {'match': {'body': q}}
    if tenant:
        query = {'bool': {'must': [query], 'filter': [{'term': {'tenant': tenant}}]}}
    return os_client.search(index=IDX, body={'size': k, '_source': NO_VECTORS, 'query': query})['hits']['hits']

def knn(v, k, tenant=None):
    if not tenant:
        return os_client.search(index=IDX, body={'size': k, '_source': NO_VECTORS,
                                                 'query': {'knn': {'embedding': {'vector': v, 'k': k}}}})['hits']['hits']
    tf = {'term': {'tenant': tenant}}
    try:      # efficient filtering: search only this tenant's vectors
        return os_client.search(index=IDX, body={'size': k, '_source': NO_VECTORS,
                                                 'query': {'knn': {'embedding': {'vector': v, 'k': k, 'filter': tf}}}})['hits']['hits']
    except Exception:   # fallback: over-fetch globally, post-filter, keep k
        return os_client.search(index=IDX, body={'size': k, '_source': NO_VECTORS, 'query': {'bool': {
            'must': [{'knn': {'embedding': {'vector': v, 'k': max(50, k * 10)}}}], 'filter': [tf]}}})['hits']['hits']

def hybrid_rrf(q, k=3, tenant=None, rrf_k=60):
    v = labkit.embed([q])[0]
    fused = {}
    for hits in (bm25(q, 10, tenant), knn(v, 10, tenant)):
        seen = []
        for h in hits:
            cid = h['_source']['clause_id']
            if cid not in seen:          # count each clause once per list (duplicates exist on Serverless)
                seen.append(cid)
                entry = fused.setdefault(cid, {**h['_source'], 'score': 0.0})
                entry['score'] += 1.0 / (rrf_k + len(seen))
    return sorted(fused.values(), key=lambda d: d['score'], reverse=True)[:k]

for q in ('piracy exclusions at sea', 'what happens if hackers lock our files'):
    print(f'\n=== {q}')
    for d in hybrid_rrf(q):
        print(f"  {d['score']:.5f}  {d['clause_id']}  {d['body'][:60]}")

**Server-side hybrid.** OpenSearch's `hybrid` query plus a *search pipeline* fuses inside the engine in one request:
- a `normalization-processor` (min-max scores, then a weighted mean);
- or, in newer versions, a `score-ranker-processor` doing RRF.

Whether your collection offers it depends on the OpenSearch version behind Serverless. The next cell tries it and tells you.

In [ ]:
PIPELINE = 'wt-hybrid'
q = 'piracy exclusions at sea'
v = labkit.embed([q])[0]
body = {'size': 3, '_source': {'excludes': ['embedding']}, 'query': {'hybrid': {'queries': [
    {'match': {'body': q}}, {'knn': {'embedding': {'vector': v, 'k': 10}}}]}}}
pipelines = [
    ('RRF (score-ranker-processor)', {'phase_results_processors': [{'score-ranker-processor': {'combination': {'technique': 'rrf'}}}]}),
    ('min-max + weighted mean (normalization-processor)', {'phase_results_processors': [{'normalization-processor': {
        'normalization': {'technique': 'min_max'},
        'combination': {'technique': 'arithmetic_mean', 'parameters': {'weights': [0.3, 0.7]}}}}]}),
]
for label, pipeline in pipelines:
    try:
        os_client.transport.perform_request('PUT', f'/_search/pipeline/{PIPELINE}', body=pipeline)
        resp = os_client.search(index=IDX, body=body, params={'search_pipeline': PIPELINE})
        print(f'server-side hybrid, {label}:')
        show(resp)
        break
    except Exception as e:
        print(f'{label}: not available here ({str(e)[:120]})')
else:
    print('\n-> use client-side RRF (above), as the lab does.')
try:
    os_client.transport.perform_request('DELETE', f'/_search/pipeline/{PIPELINE}')
except Exception:
    pass

## 8 · Agentic pattern: a retrieval tool built for grounding

When an agent calls a retrieval tool, the tool's output becomes the model's evidence. Design it for that:
- **Return citations, not just text:** stable ids (`clause_id`), the line of business, a score and a short snippet. The answer can then cite *CP-01*, and you can audit it later.
- **Bound the request.** Clamp `k`, and truncate snippets, so the model can't pull the whole index into its context.
- **Take security filters from trusted context.** The tenant comes from the authenticated session, *not* from a tool parameter the model could change.

The tool below is registered on a scratch MCP server, the same mechanism as goal 1. Look at its input schema: **there is no `tenant` parameter**.

In [ ]:
from mcp.server.fastmcp import FastMCP

def build_server(tenant):
    """One server per authenticated session; `tenant` comes from the caller's verified identity."""
    server = FastMCP(f'clauses-{tenant}')

    @server.tool()
    def search_clauses(question: str, k: int = 3) -> str:
        """Search this broker's insurance clauses. Returns clause ids, lines of business, scores and short snippets to cite."""
        k = max(1, min(int(k), 5))
        hits = hybrid_rrf(question, k=k, tenant=tenant)
        return json.dumps([{'clause_id': d['clause_id'], 'lob': d['lob'], 'score': round(d['score'], 5),
                            'snippet': d['body'][:160]} for d in hits], indent=2)
    return server, search_clauses

server, search_clauses = build_server('harbor')
for t in await server.list_tools():
    print('tool:', t.name, '\ninput schema:', json.dumps(t.inputSchema, indent=2))
print('\nharbor asks about exclusions:')
print(search_clauses('What is excluded from cover?', k=10))      # k is clamped to 5

### How this maps to Bedrock Knowledge Bases

A Knowledge Base with an OpenSearch Serverless vector store does what sections 2–7 did by hand:
- It creates a faiss HNSW index with three fields: a vector field, a text-chunk field (`AMAZON_BEDROCK_TEXT_CHUNK`) and a metadata field (`AMAZON_BEDROCK_METADATA`).
- It chunks and embeds your S3 documents.
- At query time, its `Retrieve` filters turn into OpenSearch filters. *Hybrid* search type runs BM25 + kNN together.

The security model is the one in section 1. The KB's service role must be in the collection's **data access policy**, and that's the most common reason a KB sync fails.

## Clean up

In [ ]:
if os_client.indices.exists(index=IDX):
    os_client.indices.delete(index=IDX)
print(f'deleted scratch index {IDX}; the lab index coa_chunks_knn is untouched')